# CDD-11 continuous-beta controller pilot

This notebook freezes A3-M and trains a 32x32-block beta controller with normalized reconstruction regret. Controller training, epoch calibration, and validation use source-disjoint scene sets (15/5/5); CDD-11_test is never loaded. The deployable controller is compared with calibrated fixed beta and the privileged continuous oracle under prespecified gates.


In [ ]:
import json, os, subprocess, zipfile
from pathlib import Path
from IPython.display import FileLink, display

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)


In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
A3M_ROOT = Path("/kaggle/working/experiments_a3m")
RUN_NAME = "a3m_multiscale_degradation_sidd32_seed42_20ep"
A3M_CHECKPOINT = A3M_ROOT / RUN_NAME / "best.pt"
OUTPUT_ROOT = Path("/kaggle/working/beta_controller")
assert CDD11_ROOT.is_dir() and PRETRAINED_ROOT.is_dir()
gpu_names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), f"Select 2xT4; found {gpu_names}"
print("GPUs:", gpu_names)


In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT), "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/beta_controller_audit.json",
], check=True)


In [ ]:
# Train A3-M only when the frozen checkpoint is absent.
if not A3M_CHECKPOINT.is_file():
    subprocess.run([
        "python", "-m", "hybrid_cot_nafnet.run_ablation",
        "--config", "configs/calibration_a3_multiscale.json",
        "--data-root", str(CDD11_ROOT), "--experiments-root", str(A3M_ROOT),
        "--nproc-per-node", "2", "--runs", RUN_NAME,
    ], check=True)
assert A3M_CHECKPOINT.is_file(), f"Missing checkpoint: {A3M_CHECKPOINT}"
print("Frozen restorer:", A3M_CHECKPOINT)


In [ ]:
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.train_beta_controller",
    "--checkpoint", str(A3M_CHECKPOINT),
    "--data-root", str(CDD11_ROOT),
    "--output-dir", str(OUTPUT_ROOT),
    "--git-commit", COMMIT,
    "--block-size", "32", "--fixed-beta", "0.99",
    "--blocks-per-image", "64", "--calibration-scenes", "5",
    "--epochs", "20", "--batch-size", "256",
], check=True)


In [ ]:
import pandas as pd
summary = json.loads((OUTPUT_ROOT / "summary.json").read_text())
display(summary)
display(pd.read_csv(OUTPUT_ROOT / "train_log.csv"))
display(pd.read_csv(OUTPUT_ROOT / "validation_metrics.csv"))
archive = Path("/kaggle/working/beta_controller_results.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            output_zip.write(path, path.relative_to(OUTPUT_ROOT.parent))
display(FileLink(str(archive)))
